In [ ]:
import json, pandas as pd
from pathlib import Path
from huggingface_hub import HfApi

TOKEN = ""  # insert  HF token here
UPLOAD = Path(".")

# Load annotations
ann = pd.read_csv("Bluesky Moderation Annotations.csv")
ann.columns = ann.columns.str.strip()
ann = ann.drop_duplicates(subset=['annotator_name','post_id'], keep='last')

# Main survey only (display_num < 10000) — both Ines and adash
main_ann = ann[ann['display_num'] < 10000].copy()

# Pilot remaining (display_num 20000-30000)
pilot_rem_ann = ann[(ann['display_num'] >= 20000) & (ann['display_num'] < 30000)].copy()

# Combine both
combined = pd.concat([main_ann, pilot_rem_ann], ignore_index=True)

# Build lookups
lu_ines  = dict(zip(combined[combined['annotator_name']=='Ines']['post_id'],
                    combined[combined['annotator_name']=='Ines']['label']))
lu_adash = dict(zip(combined[combined['annotator_name']=='adash']['post_id'],
                    combined[combined['annotator_name']=='adash']['label']))

# Find posts labeled by both where source is pilot
# Load combined_3k to get source info
main_posts = {}
with open(UPLOAD / "combined_3k.jsonl") as f:
    for line in f:
        if line.strip():
            p = json.loads(line)
            if p.get('uri'): main_posts[p['uri']] = p

# Also load pilot_1k for pilot post metadata
pilot_posts = {}
with open(UPLOAD / "pilot_1k.jsonl") as f:
    for line in f:
        if line.strip():
            p = json.loads(line)
            if p.get('uri'): pilot_posts[p['uri']] = p

# Load pilot_remaining
pilot_rem_posts = {}
with open(UPLOAD / "pilot_remaining.jsonl") as f:
    for line in f:
        if line.strip():
            p = json.loads(line)
            if p.get('uri'): pilot_rem_posts[p['uri']] = p

all_pilot_posts = {**pilot_posts, **pilot_rem_posts}
print(f"Total pilot posts: {len(all_pilot_posts)}")

# Find pilot disagreements (Ines vs adash)
common = set(lu_ines) & set(lu_adash)
pilot_common = [p for p in common if p in all_pilot_posts]
pilot_disagree = [p for p in pilot_common if lu_ines[p] != lu_adash[p]]

print(f"Pilot posts labeled by both: {len(pilot_common)}")
print(f"Pilot disagreements: {len(pilot_disagree)}")

# Get reasons and categories
cats_ines  = dict(zip(combined[combined['annotator_name']=='Ines']['post_id'],
                      combined[combined['annotator_name']=='Ines']['unsafe_categories'].fillna('')))
cats_adash = dict(zip(combined[combined['annotator_name']=='adash']['post_id'],
                      combined[combined['annotator_name']=='adash']['unsafe_categories'].fillna('')))

# Build JSONL
out_rows = []
for uri in pilot_disagree:
    post = dict(all_pilot_posts[uri])
    post['ines_label']       = lu_ines[uri]
    post['adash_label']      = lu_adash[uri]
    post['ines_categories']  = cats_ines.get(uri, '')
    post['adash_categories'] = cats_adash.get(uri, '')
    post['source']           = 'pilot_disagree'
    out_rows.append(post)

print(f"\nBuilding JSONL with {len(out_rows)} posts")
print(f"Ines→Unsafe, adash→Safe: {sum(1 for u in pilot_disagree if lu_ines[u]=='Unsafe')}")
print(f"Ines→Safe, adash→Unsafe: {sum(1 for u in pilot_disagree if lu_adash[u]=='Unsafe')}")

out_path = UPLOAD / "data/pilot_disagreements.jsonl"
with open(out_path, 'w') as f:
    for p in out_rows:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')
print(f"Saved → {out_path}")

# Upload
api = HfApi(token=TOKEN)
api.upload_file(
    path_or_fileobj=str(out_path),
    path_in_repo="pilot_disagreements.jsonl",
    repo_id="ines-abdelaziz/neurips-content-moderation",
    repo_type="dataset",
    token=TOKEN,
)
print("Uploaded to HF!")

Total pilot posts: 1000
Pilot posts labeled by both: 573
Pilot disagreements: 109

Building JSONL with 109 posts
Ines→Unsafe, adash→Safe: 35
Ines→Safe, adash→Unsafe: 74
Saved → data/pilot_disagreements.jsonl
Uploaded to HF!


In [7]:
import json, pandas as pd

ann = pd.read_csv("Bluesky Moderation Annotations.csv")
ann.columns = ann.columns.str.strip()

lu_ines = set(ann[ann["annotator_name"] == "Ines"]["post_id"])
lu_adash = set(ann[ann["annotator_name"] == "adash"]["post_id"])

uris = []
with open("top_unlabeled_neighbors_1000.jsonl") as f:
    for line in f:
        if line.strip():
            uris.append(json.loads(line)["query"]["uri"])

only_ines = [u for u in uris if u in lu_ines and u not in lu_adash]
only_adash = [u for u in uris if u in lu_adash and u not in lu_ines]
neither = [u for u in uris if u not in lu_ines and u not in lu_adash]
both = [u for u in uris if u in lu_ines and u in lu_adash]

print(f"Both labeled:      {len(both)}")
print(f"Only Ines:         {len(only_ines)}")
print(f"Only adash:        {len(only_adash)}")
print(f"Neither labeled:   {len(neither)}")

Both labeled:      1000
Only Ines:         0
Only adash:        0
Neither labeled:   0


In [9]:
import json, pandas as pd

ann = pd.read_csv("Bluesky Moderation Annotations.csv")
ann.columns = ann.columns.str.strip()

# Only automod block
automod_uris = set(ann[ann["display_num"] >= 40000]["post_id"])
print(f"Automod labeled URIs: {len(automod_uris)}")

uris = []
with open("top_unlabeled_neighbors_1000.jsonl") as f:
    for line in f:
        if line.strip():
            uris.append(json.loads(line)["query"]["uri"])

already = [u for u in uris if u in automod_uris]
remaining = [u for u in uris if u not in automod_uris]
print(f"Already labeled (automod): {len(already)}")
print(f"Remaining to label:        {len(remaining)}")

Automod labeled URIs: 451
Already labeled (automod): 163
Remaining to label:        837


In [13]:
import json, pandas as pd

ann = pd.read_csv("Bluesky Moderation Annotations.csv")
ann.columns = ann.columns.str.strip()

automod_uris = set(ann[ann["display_num"] >= 40000]["post_id"])

uris = []
with open("top_unlabeled_neighbors_1000.jsonl") as f:
    for line in f:
        if line.strip():
            uris.append(json.loads(line)["query"]["uri"])

remaining = [u for u in uris if u not in automod_uris]
print(f"Remaining (not in automod): {len(remaining)}")

# Check which other surveys they appear in
ann_remaining = ann[ann["post_id"].isin(remaining)]
print(f"\nOf those 837, how many appear in other surveys:")
blocks = {
    "main (<10000)": ann_remaining[ann_remaining["display_num"] < 10000],
    "firehose (10-20K)": ann_remaining[
        (ann_remaining["display_num"] >= 10000) & (ann_remaining["display_num"] < 20000)
    ],
    "pilot_rem (20-30K)": ann_remaining[
        (ann_remaining["display_num"] >= 20000) & (ann_remaining["display_num"] < 30000)
    ],
    "pilot_dis (30-40K)": ann_remaining[
        (ann_remaining["display_num"] >= 30000) & (ann_remaining["display_num"] < 40000)
    ],
}
for name, block in blocks.items():
    uq = block["post_id"].nunique()
    if uq:
        print(f"  {name}: {uq} unique posts")

# Truly never labeled
all_labeled = set(ann["post_id"])
truly_new = [u for u in remaining if u not in all_labeled]
print(f"\nTruly never labeled (not in any survey): {len(truly_new)}")

Remaining (not in automod): 837

Of those 837, how many appear in other surveys:
  main (<10000): 213 unique posts
  pilot_rem (20-30K): 624 unique posts

Truly never labeled (not in any survey): 0


In [16]:
import json, pandas as pd

ann = pd.read_csv("Bluesky Moderation Annotations.csv")
ann.columns = ann.columns.str.strip()
ann = ann.drop_duplicates(subset=["annotator_name", "post_id"], keep="last")

lu_ines = set(ann[ann["annotator_name"] == "Ines"]["post_id"])
lu_adash = set(ann[ann["annotator_name"] == "adash"]["post_id"])

match_uris = []
with open("top_unlabeled_neighbors_1000.jsonl") as f:
    for line in f:
        if line.strip():
            row = json.loads(line)
            uri = row["match"].get("uri", "")
            if uri:
                match_uris.append(uri)

print(f"Total match URIs: {len(match_uris)}")
print(f"Unique match URIs: {len(set(match_uris))}")

already_ines = [u for u in match_uris if u in lu_ines]
already_adash = [u for u in match_uris if u in lu_adash]
both = [u for u in set(match_uris) if u in lu_ines and u in lu_adash]
neither = [u for u in set(match_uris) if u not in lu_ines and u not in lu_adash]

print(f"\nAlready labeled by Ines:  {len(already_ines)}")
print(f"Already labeled by adash: {len(already_adash)}")
print(f"Both labeled:             {len(both)}")
print(f"Neither labeled:          {len(neither)}")

Total match URIs: 1000
Unique match URIs: 988

Already labeled by Ines:  292
Already labeled by adash: 292
Both labeled:             288
Neither labeled:          700


In [17]:

import json
with open('top_unlabeled_neighbors_1000.jsonl') as f:
    row = json.loads(f.readline())
print('match keys:', list(row['match'].keys()))
print('match sample:', {k:str(v)[:80] for k,v in row['match'].items()})


match keys: ['cid', 'uri', 'did', 'text', 'record', 'image_file', 'alt_text', 'labels']
match sample: {'cid': 'bafyreia2pedf5fqavpwnoii3gssxyt4xhdjesfm3am5wvxwqhxzww7i5o4', 'uri': 'at://did:plc:tryxpttftqtm5oblixabevle/app.bsky.feed.post/3lojepk4tkk2z', 'did': 'did:plc:tryxpttftqtm5oblixabevle', 'text': 'you seem like a real piece of shit', 'record': "{'text': 'you seem like a real piece of shit', '$type': 'app.bsky.feed.post', 'l", 'image_file': 'None', 'alt_text': 'None', 'labels': '[]'}


In [18]:
import json

paths = []
with open("top_unlabeled_neighbors_1000.jsonl") as f:
    for line in f:
        if line.strip():
            row = json.loads(line)
            img = row["match"].get("image_file", "")
            if img and img != "None":
                paths.append(img)

print(f"Posts with images: {len(paths)}")
print("\nSample paths:")
for p in paths[:5]:
    print(f"  {p}")

Posts with images: 662

Sample paths:
  /NS/decentralized-social-media_dataset1/nobackup/dataset_nsfw_firehose_pds_all/images/bafyreiasn2twwdz6yywcm4dhvan6wjtpfsj2iuipbrrmr5mzp6sy6hqfvu.jpeg
  /NS/decentralized-social-media_dataset1/nobackup/dataset_nsfw_firehose_pds_all/images/bafyreieu4wdvsqapvrt2ot64um7wyh7owfcv2i7673cgl5nmiud3zz7yni.jpeg
  /NS/decentralized-social-media_dataset1/nobackup/dataset_nsfw_firehose_pds_all/images/bafyreibscvuo3xhy64uhj5siyddbotdfhrmf7kopkvdio5r2xcrhb56fmm.jpeg
  /NS/decentralized-social-media_dataset1/nobackup/dataset_nsfw_firehose_pds_all/images/bafyreie2zhlstrptfapmsxse2yhcyxxoiqaopcd5dqykokcon5dpf3hboe.jpeg
  /NS/decentralized-social-media_dataset1/nobackup/dataset_nsfw_firehose_pds_all/images/bafyreidz7wiccdcrek2i7vikmlk7skcu2pko6ipzw27o4miw4owkmolspq.jpeg
